<p style="font-size:32px; font-weight: bold;">Librerias.</p>

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from scipy.stats import friedmanchisquare, wilcoxon
from statistics import mean
from scipy import stats
from statsmodels.stats.multitest import multipletests
import scikit_posthocs as sp
import numpy as np
import shap
import pprint
import time

<p style="font-size:32px; font-weight: bold;">Preprocesado.</p>

In [4]:
data = pd.read_excel("Planilla datos Totales 2021_ 2022 (528 casos).xlsx")

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 528 entries, 0 to 527
Data columns (total 25 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   ID niño/a        528 non-null    object 
 1   Edad (meses)     528 non-null    int64  
 2   Edad (años)      528 non-null    float64
 3   Sexo             528 non-null    int64  
 4   Curso            528 non-null    int64  
 5   Tipo EE          528 non-null    int64  
 6   Depen. EE        528 non-null    int64  
 7   Cohorte          528 non-null    int64  
 8   MTV_WM           528 non-null    int64  
 9   MTVE_ Torpo      522 non-null    float64
 10  INHCon_Bzz       525 non-null    float64
 11  INHCog_Stroop    528 non-null    int64  
 12  FC_DCCS          528 non-null    int64  
 13  PLA_Porteus      528 non-null    int64  
 14  Com_TEMT         528 non-null    int64  
 15  Clas_TEMT        528 non-null    int64  
 16  Corres_TEMT      528 non-null    int64  
 17  Seria_TEMT      

In [6]:
data = data.drop(data.columns[[0] + list(range(13, 22))], axis=1)

In [7]:
data = data.fillna(0)

In [8]:
data = data.drop('Edad (meses)', axis=1)

In [9]:
data

,Edad (años),Sexo,Curso,Tipo EE,Depen. EE,Cohorte,MTV_WM,MTVE_ Torpo,INHCon_Bzz,INHCog_Stroop,FC_DCCS,CMLR_TEMT,CMN_TEMT,CMG_TEMT
0,5.750000,1,2,2,2,1,9,16.0,17.0,35,16,18,10,2
1,5.416667,1,2,2,2,1,7,15.0,17.0,41,12,17,9,2
2,5.916667,2,2,2,2,1,5,4.0,17.0,11,9,15,14,1
3,4.583333,2,1,2,2,1,0,5.0,19.0,20,9,10,3,2
4,5.250000,1,1,2,2,1,6,4.0,17.0,25,13,16,9,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
523,6.500000,2,2,2,2,2,0,2.0,17.0,30,8,16,12,3
524,5.910000,2,2,2,2,2,2,6.0,17.0,7,16,14,13,2
525,6.410000,2,2,2,2,2,1,7.0,2.0,18,12,16,12,3
526,6.500000,2,2,2,2,2,2,5.0,17.0,16,16,16,16,2


In [10]:
# Features (X) y etiquetas (y)
X = data.drop(data.columns[-3:], axis=1)  # Features, excluyendo las últimas 3 columnas
y = data.iloc[:, -3:]  # Últimas 3 columnas, que son las etiquetas


In [11]:
resultsagg_cdt_cat = pd.DataFrame()
resultsagg_cat = pd.DataFrame()

In [12]:
columnas = ["Edad (años)", "Sexo", "Curso", "Tipo EE", "Depen. EE", "Cohorte", "MTV_WM", 
            "MTVE_ Torpo", "INHCon_Bzz", "INHCog_Stroop", "FC_DCCS"]

<p style="font-size:32px; font-weight: bold;">Modelo CatBoost por target</p>

In [14]:
#CatBoostRegressor(iterations=500, depth=5, learning_rate=0.01, l2_leaf_reg=1, verbose=0)
#XGBRegressor(objective="reg:squarederror", colsample_bytree = 1, subsample = 0.8,n_estimators = 300, learning_rate = 0.01, max_depth = 3, eval_metric='rmse')
#LGBMRegressor(n_estimators=250, max_depth=4, learning_rate=0.01, num_leaves=60)
cat_model = CatBoostRegressor(iterations=500, depth=5, learning_rate=0.01, l2_leaf_reg=1, verbose=0)
permutation_importance_cat_CMLR = []
mse_targets_CMLR = []
permutation_importance_orden_cat_CMLR = []
permutation_cat_cmlr = []

for i in range(50):
    # Dividir el dataset para el target actual
    X_train, X_test, y_train, y_test = train_test_split(X, y["CMLR_TEMT"], test_size=0.15, random_state=i)
    X_train_2, X_val, y_train_2, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=i)
        
    # Combinar conjuntos de entrenamiento y validación
    X_combined = pd.concat([X_train_2, X_val], axis=0)
    y_combined = pd.concat([y_train_2, y_val], axis=0)
    cat_model.fit(X_combined, y_combined)
        
    # Realizar predicción y calcular el MSE
    y_pred = cat_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_targets_CMLR.append(mse)
        
    # If you want to sort the features by importance
    result = permutation_importance(cat_model, X_test, y_test, n_repeats=20)
    perm_imp = result.importances_mean
    
    sorted_indices = np.argsort(perm_imp)[::-1]
    sorted_importances = perm_imp[sorted_indices]
    sorted_features = X_train.columns[sorted_indices]
    
    permutation_importance_cat_CMLR.append(perm_imp)
    permutation_importance_orden_cat_CMLR.append(sorted_importances)


resultsagg_cdt_cat["CMLR"] = mse_targets_CMLR

In [15]:
#CatBoostRegressor(iterations=500, depth=5, learning_rate=0.01, l2_leaf_reg=1, verbose=0)
#XGBRegressor(objective="reg:squarederror", colsample_bytree = 1, subsample = 0.8,n_estimators = 300, learning_rate = 0.01, max_depth = 3, eval_metric='rmse')
cat_model = CatBoostRegressor(iterations=500, depth=5, learning_rate=0.01, l2_leaf_reg=1, verbose=0)
permutation_importance_cat_CMN = []
mse_targets_CMN = []
permutation_importance_orden_cat_CMN = []
permutation_cat_cmn = []

for i in range(50):
    # Dividir el dataset para el target actual
    X_train, X_test, y_train, y_test = train_test_split(X, y["CMN_TEMT"], test_size=0.15, random_state=i)
    X_train_2, X_val, y_train_2, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=i)
        
    # Combinar conjuntos de entrenamiento y validación
    X_combined = pd.concat([X_train_2, X_val], axis=0)
    y_combined = pd.concat([y_train_2, y_val], axis=0)
    cat_model.fit(X_combined, y_combined)
        
    # Realizar predicción y calcular el MSE
    y_pred = cat_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_targets_CMN.append(mse)
        
    # If you want to sort the features by importance
    result = permutation_importance(cat_model, X_test, y_test, n_repeats=20)
    perm_imp = result.importances_mean
    
    sorted_indices = np.argsort(perm_imp)[::-1]
    sorted_importances = perm_imp[sorted_indices]
    sorted_features = X_train.columns[sorted_indices]

    permutation_importance_cat_CMN.append(perm_imp)
    permutation_importance_orden_cat_CMN.append(sorted_importances)

    
resultsagg_cdt_cat["CMN"] = mse_targets_CMN

In [16]:
cat_model = CatBoostRegressor(iterations=500, depth=5, learning_rate=0.01, l2_leaf_reg=1, verbose=0)
permutation_importance_cat_CMG = []
mse_targets_CMG = []
permutation_cat_cmg = []

for i in range(50):
    # Dividir el dataset para el target actual
    X_train, X_test, y_train, y_test = train_test_split(X, y["CMG_TEMT"], test_size=0.15, random_state=i)
    X_train_2, X_val, y_train_2, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=i)
        
    # Combinar conjuntos de entrenamiento y validación
    X_combined = pd.concat([X_train_2, X_val], axis=0)
    y_combined = pd.concat([y_train_2, y_val], axis=0)
    cat_model.fit(X_combined, y_combined)
        
    # Realizar predicción y calcular el MSE
    y_pred = cat_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_targets_CMG.append(mse)
        
    # If you want to sort the features by importance
    result = permutation_importance(cat_model, X_test, y_test, n_repeats=20)
    perm_imp = result.importances_mean
    
    sorted_indices = np.argsort(perm_imp)[::-1]
    sorted_importances = perm_imp[sorted_indices]
    sorted_features = X_train.columns[sorted_indices]

    permutation_importance_cat_CMG.append(perm_imp)


resultsagg_cdt_cat["CMG"] = mse_targets_CMG

In [17]:
features_rf_CMLR = pd.DataFrame(data=permutation_importance_cat_CMLR, columns= columnas)
features_rf_CMLR.describe()

,Edad (años),Sexo,Curso,Tipo EE,Depen. EE,Cohorte,MTV_WM,MTVE_ Torpo,INHCon_Bzz,INHCog_Stroop,FC_DCCS
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,0.041153,0.002797,0.016848,0.010848,0.018349,0.002748,0.295071,0.013561,0.002565,0.017808,0.074663
std,0.024419,0.006296,0.012028,0.008660,0.014045,0.003765,0.037382,0.014883,0.007152,0.014632,0.023397
min,-0.019219,-0.015214,-0.020268,-0.014877,-0.025242,-0.013143,0.207154,-0.033391,-0.012534,-0.011910,-0.004863
25%,0.022648,-0.000958,0.011250,0.006481,0.011082,0.000562,0.266515,0.002862,-0.001962,0.005914,0.060646
50%,0.042085,0.002864,0.017173,0.013366,0.020266,0.003404,0.298942,0.015492,0.002474,0.018856,0.075029
75%,0.060607,0.007319,0.023296,0.015621,0.025912,0.005168,0.323082,0.024761,0.007596,0.028605,0.090561
max,0.088431,0.012616,0.041461,0.034175,0.047933,0.010514,0.368478,0.037404,0.022019,0.040405,0.123280


In [18]:
features_cat_CMN = pd.DataFrame(data=permutation_importance_cat_CMN, columns= columnas)
features_cat_CMN.describe()

,Edad (años),Sexo,Curso,Tipo EE,Depen. EE,Cohorte,MTV_WM,MTVE_ Torpo,INHCon_Bzz,INHCog_Stroop,FC_DCCS
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,0.047806,-0.001159,0.015363,0.012967,0.025876,0.000710,0.416745,0.015108,-0.000411,0.010504,0.043240
std,0.019468,0.002564,0.009706,0.009252,0.012053,0.001636,0.046593,0.011502,0.005487,0.010129,0.016033
min,-0.015813,-0.009185,-0.010599,-0.012876,0.003647,-0.002854,0.296650,-0.012621,-0.011778,-0.030013,-0.003343
25%,0.034748,-0.002371,0.007614,0.007524,0.018034,-0.000500,0.389665,0.005143,-0.004126,0.005038,0.034134
50%,0.049899,-0.000912,0.017709,0.012432,0.026508,0.000685,0.419974,0.017200,-0.000037,0.011576,0.043390
75%,0.061267,0.000769,0.023070,0.018074,0.033859,0.002092,0.442464,0.022762,0.003372,0.017285,0.053646
max,0.082313,0.003374,0.028569,0.037472,0.049944,0.003509,0.506810,0.033984,0.010931,0.029447,0.084151


In [19]:
features_cat_CMG = pd.DataFrame(data=permutation_importance_cat_CMG, columns= columnas)
features_cat_CMG.describe()

,Edad (años),Sexo,Curso,Tipo EE,Depen. EE,Cohorte,MTV_WM,MTVE_ Torpo,INHCon_Bzz,INHCog_Stroop,FC_DCCS
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,0.377646,-0.001559,-0.000629,0.024080,0.023222,0.001073,0.340989,0.024161,0.004332,0.001655,0.066261
std,0.084472,0.003782,0.006793,0.015209,0.017263,0.003418,0.064254,0.022665,0.009725,0.009601,0.029110
min,0.178978,-0.011937,-0.026076,-0.018596,-0.021786,-0.007168,0.232600,-0.015938,-0.019603,-0.020956,0.004929
25%,0.325166,-0.003343,-0.003347,0.014500,0.009628,-0.000627,0.286342,0.005394,-0.000440,-0.004246,0.045109
50%,0.373410,-0.001450,0.000605,0.025523,0.022898,0.001772,0.328468,0.022888,0.004761,0.003103,0.067347
75%,0.449767,0.001444,0.004104,0.031136,0.036552,0.003056,0.386682,0.040368,0.010384,0.007356,0.091554
max,0.524617,0.005215,0.010509,0.051879,0.051476,0.006627,0.504603,0.071611,0.029441,0.025156,0.113077


<p style="font-size:26px; font-weight: bold;">Resultados CatBoost.</p>

In [21]:
promedio_CMLR = np.mean(permutation_importance_cat_CMLR, axis=0)
promedio_CMG = np.mean(permutation_importance_cat_CMG, axis=0)
promedio_CMN = np.mean(permutation_importance_cat_CMN, axis=0)

In [22]:
# Convertir cada lista a un array de NumPy. La forma resultante será (50, 12)
arr1 = np.array(permutation_importance_cat_CMLR)
arr2 = np.array(permutation_importance_cat_CMG)
arr3 = np.array(permutation_importance_cat_CMN)

# Sumar los arrays de forma element-wise y dividir entre 3
promedio_global = (arr1 + arr2 + arr3) / 3

promedio_final = np.mean(promedio_global, axis=0)

In [23]:
len(promedio_global)

50

In [24]:
# Agregar las importancias en un ranking global: se promedian las importancias de los 3 targets para cada feature
importancia_global = (promedio_CMLR + promedio_CMG + promedio_CMN) / 3

print("Importancia promedio target CMLR:", promedio_CMLR)
print("Importancia promedio target CMG:", promedio_CMG)
print("Importancia promedio target CMN:", promedio_CMN)
print("Importancia global agregada:", importancia_global)

Importancia promedio target CMLR: [0.04115315 0.00279728 0.01684847 0.010848   0.01834881 0.00274751
 0.29507129 0.01356089 0.00256486 0.01780771 0.07466285]
Importancia promedio target CMG: [ 0.37764648 -0.00155914 -0.00062881  0.02408002  0.02322244  0.00107328
  0.34098893  0.02416111  0.00433247  0.00165469  0.06626098]
Importancia promedio target CMN: [ 4.78062621e-02 -1.15911315e-03  1.53626519e-02  1.29674865e-02
  2.58763885e-02  7.10000237e-04  4.16744558e-01  1.51078635e-02
 -4.10768776e-04  1.05040919e-02  4.32399178e-02]
Importancia global agregada: [1.55535299e-01 2.63405143e-05 1.05274390e-02 1.59651690e-02
 2.24825478e-02 1.51026230e-03 3.50934926e-01 1.76099551e-02
 2.16218384e-03 9.98883078e-03 6.13879148e-02]


In [25]:
# Convertir las importancias a rankings
# Se utiliza rank con ascending=False, pues se asume que un valor mayor de importancia implica mejor posición.
ranking_CMLR = pd.Series(promedio_CMLR).rank(ascending=False).values
ranking_CMG = pd.Series(promedio_CMG).rank(ascending=False).values
ranking_CMN = pd.Series(promedio_CMN).rank(ascending=False).values
ranking_global = pd.Series(importancia_global).rank(ascending=False).values

print("Edad (años), Sexo, Curso, Tipo EE, Depen. EE, Cohorte, MTV_WM, MTVE_ Torpo, INHCon_Bzz, INHCog_Stroop, FC_DCCS")
print("\nRanking target CMLR:", ranking_CMLR)
print("Ranking target CMN:", ranking_CMN)
print("Ranking target CMG:", ranking_CMG)
print("Ranking global:", ranking_global)

# Paso 3: Validación estadística con el Test de Friedman
# El test evalúa si existen diferencias significativas entre los rankings obtenidos de cada target.
# Aquí, cada array (ranking_A, ranking_B, ranking_C) representa las posiciones asignadas a cada feature.
stat, p_value = friedmanchisquare(ranking_CMLR, ranking_CMG, ranking_CMN)
print("\nTest de Friedman:")
print("Estadístico =", stat)
print("p-valor =", p_value)

Edad (años), Sexo, Curso, Tipo EE, Depen. EE, Cohorte, MTV_WM, MTVE_ Torpo, INHCon_Bzz, INHCog_Stroop, FC_DCCS

Ranking target CMLR: [ 3.  9.  6.  8.  4. 10.  1.  7. 11.  5.  2.]
Ranking target CMN: [ 2. 11.  5.  7.  4.  9.  1.  6. 10.  8.  3.]
Ranking target CMG: [ 1. 11. 10.  5.  6.  9.  2.  4.  7.  8.  3.]
Ranking global: [ 2. 11.  7.  6.  4. 10.  1.  5.  9.  8.  3.]

Test de Friedman:
Estadístico = 0.21052631578947967
p-valor = 0.9000876262522566


<p style="font-size:20px; font-weight: bold;">Wilconxon CatBoost CMLR.</p>

In [27]:
stat, p_valor = wilcoxon(features_rf_CMLR["MTV_WM"], features_rf_CMLR["FC_DCCS"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [28]:
stat, p_valor = wilcoxon(features_rf_CMLR["Edad (años)"], features_rf_CMLR["FC_DCCS"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 93.0
p-valor: 4.8571529021046445e-09


In [29]:
stat, p_valor = wilcoxon(features_rf_CMLR["Edad (años)"], features_rf_CMLR["Depen. EE"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 151.0
p-valor: 4.2698192537216073e-07


In [30]:
stat, p_valor = wilcoxon(features_rf_CMLR["INHCog_Stroop"], features_rf_CMLR["Depen. EE"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 594.0
p-valor: 0.6806447467771797


In [31]:
stat, p_valor = wilcoxon(features_rf_CMLR["INHCog_Stroop"], features_rf_CMLR["Curso"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 570.0
p-valor: 0.5210251168620434


<p style="font-size:20px; font-weight: bold;">Wilconxon CatBoost CMN.</p>

In [33]:
stat, p_valor = wilcoxon(features_cat_CMN["MTV_WM"], features_cat_CMN["Edad (años)"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [34]:
stat, p_valor = wilcoxon(features_cat_CMN["Edad (años)"], features_cat_CMN["FC_DCCS"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 473.0
p-valor: 0.11393994202132163


In [35]:
stat, p_valor = wilcoxon(features_cat_CMN["Depen. EE"], features_cat_CMN["FC_DCCS"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 178.0
p-valor: 2.36936675790389e-06


In [36]:
stat, p_valor = wilcoxon(features_cat_CMN["Depen. EE"], features_cat_CMN["Curso"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 219.0
p-valor: 2.3068974657647345e-05


In [37]:
stat, p_valor = wilcoxon(features_cat_CMN["Curso"], features_cat_CMN["MTVE_ Torpo"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 608.0
p-valor: 0.7813934753886187


<p style="font-size:20px; font-weight: bold;">Wilconxon CatBoost CMG.</p>

In [39]:
stat, p_valor = wilcoxon(features_cat_CMG["MTV_WM"], features_cat_CMG["Edad (años)"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 393.0
p-valor: 0.017575449774248852


In [40]:
stat, p_valor = wilcoxon(features_cat_CMG["MTV_WM"], features_cat_CMG["FC_DCCS"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [41]:
stat, p_valor = wilcoxon(features_cat_CMG["FC_DCCS"], features_cat_CMG["Tipo EE"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 48.0
p-valor: 4.3806736016449577e-11


In [42]:
stat, p_valor = wilcoxon(features_cat_CMG["Tipo EE"], features_cat_CMG["Depen. EE"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 613.0
p-valor: 0.81837027658068


In [43]:
stat, p_valor = wilcoxon(features_cat_CMG["MTVE_ Torpo"], features_cat_CMG["Depen. EE"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 631.0
p-valor: 0.9542249738357711


In [44]:
# Aplicamos la corrección de Bonferroni.
# multipletests multiplica cada p-valor por el número de comparaciones o ajusta el nivel alpha.
p_values = [1.7763568394002505e-15, 7.237197507947712e-10, 1.206143563692308e-06, 0.8109402812359416, 0.3325668996889135, 1.7763568394002505e-15,
           0.10525460779046192, 1.6455253124547653e-06, 2.3068974657647345e-05, 0.8407540567937097, 0.013686934524772099, 1.7763568394002505e-15,
            4.792291008470784e-10, 0.9313837099567692, 0.7089524432753507]

rej, pvals_corr, alphacSidak, alphacBonf = multipletests(p_values, alpha=0.05, method='bonferroni')

print("\nP-valores ajustados con Bonferroni:")
print(pvals_corr)

print("\nHipótesis nula rechazada (True significa que se rechaza):")
print(rej)


P-valores ajustados con Bonferroni:
[2.66453526e-14 1.08557963e-08 1.80921535e-05 1.00000000e+00
 1.00000000e+00 2.66453526e-14 1.00000000e+00 2.46828797e-05
 3.46034620e-04 1.00000000e+00 2.05304018e-01 2.66453526e-14
 7.18843651e-09 1.00000000e+00 1.00000000e+00]

Hipótesis nula rechazada (True significa que se rechaza):
[ True  True  True False False  True False  True  True False False  True
  True False False]


<p style="font-size:32px; font-weight: bold;">Modelo CatBoost parcimonioso.</p>

In [46]:
data = data.drop(columns=['Sexo','INHCon_Bzz', 'Cohorte', 'Curso'], axis=1)

In [47]:
columnas = ["Edad (años)", 'Curso', "Tipo EE", "Depen. EE", "MTV_WM", 'MTVE_ Torpo',"FC_DCCS"]

In [48]:
data

,Edad (años),Tipo EE,Depen. EE,MTV_WM,MTVE_ Torpo,INHCog_Stroop,FC_DCCS,CMLR_TEMT,CMN_TEMT,CMG_TEMT
0,5.750000,2,2,9,16.0,35,16,18,10,2
1,5.416667,2,2,7,15.0,41,12,17,9,2
2,5.916667,2,2,5,4.0,11,9,15,14,1
3,4.583333,2,2,0,5.0,20,9,10,3,2
4,5.250000,2,2,6,4.0,25,13,16,9,2
...,...,...,...,...,...,...,...,...,...,...
523,6.500000,2,2,0,2.0,30,8,16,12,3
524,5.910000,2,2,2,6.0,7,16,14,13,2
525,6.410000,2,2,1,7.0,18,12,16,12,3
526,6.500000,2,2,2,5.0,16,16,16,16,2


In [49]:
# Features (X) y etiquetas (y)
X = data.drop(data.columns[-3:], axis=1)  # Features, excluyendo las últimas 3 columnas
y = data.iloc[:, -3:]  # Últimas 3 columnas, que son las etiquetas

In [50]:
resultsagg_cat_p = pd.DataFrame()

In [51]:
cat_model = CatBoostRegressor(iterations=500, depth=5, learning_rate=0.01, l2_leaf_reg=1, verbose=0)
permutation_importance_cat_CMLR_p = []
mse_targets_CMLR_p = []
permutation_importance_orden_cat_CMLR_p = []
permutation_cat_cmlr_p = []

for i in range(50):
    # Dividir el dataset para el target actual
    X_train, X_test, y_train, y_test = train_test_split(X, y["CMLR_TEMT"], test_size=0.15, random_state=i)
    X_train_2, X_val, y_train_2, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=i)
        
    # Combinar conjuntos de entrenamiento y validación
    X_combined = pd.concat([X_train_2, X_val], axis=0)
    y_combined = pd.concat([y_train_2, y_val], axis=0)
    cat_model.fit(X_combined, y_combined)
        
    # Realizar predicción y calcular el MSE
    y_pred = cat_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_targets_CMLR_p.append(mse)
        
    # If you want to sort the features by importance
    result = permutation_importance(cat_model, X_test, y_test, n_repeats=20)
    perm_imp = result.importances_mean
    
    sorted_indices = np.argsort(perm_imp)[::-1]
    sorted_importances = perm_imp[sorted_indices]
    sorted_features = X_train.columns[sorted_indices]
    
    permutation_importance_cat_CMLR_p.append(perm_imp)
    permutation_importance_orden_cat_CMLR_p.append(sorted_importances)


resultsagg_cat_p["CMLR"] = mse_targets_CMLR_p

In [52]:
cat_model = CatBoostRegressor(iterations=500, depth=5, learning_rate=0.01, l2_leaf_reg=1, verbose=0)
permutation_importance_cat_CMN_p = []
mse_targets_CMN_p = []
permutation_importance_orden_cat_CMN_p = []
permutation_cat_cmn_p = []

for i in range(50):
    # Dividir el dataset para el target actual
    X_train, X_test, y_train, y_test = train_test_split(X, y["CMN_TEMT"], test_size=0.15, random_state=i)
    X_train_2, X_val, y_train_2, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=i)
        
    # Combinar conjuntos de entrenamiento y validación
    X_combined = pd.concat([X_train_2, X_val], axis=0)
    y_combined = pd.concat([y_train_2, y_val], axis=0)
    cat_model.fit(X_combined, y_combined)
        
    # Realizar predicción y calcular el MSE
    y_pred = cat_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_targets_CMN_p.append(mse)
        
    # If you want to sort the features by importance
    result = permutation_importance(cat_model, X_test, y_test, n_repeats=20)
    perm_imp = result.importances_mean
    
    sorted_indices = np.argsort(perm_imp)[::-1]
    sorted_importances = perm_imp[sorted_indices]
    sorted_features = X_train.columns[sorted_indices]

    permutation_importance_cat_CMN_p.append(perm_imp)
    permutation_importance_orden_cat_CMN_p.append(sorted_importances)

    
resultsagg_cat_p["CMN"] = mse_targets_CMN_p

In [53]:
cat_model = CatBoostRegressor(iterations=500, depth=5, learning_rate=0.01, l2_leaf_reg=1, verbose=0)
permutation_importance_cat_CMG_p = []
mse_targets_CMG_p = []
permutation_cat_cmg_p = []
permutation_importance_orden_cat_CMG_p = []
for i in range(50):
    # Dividir el dataset para el target actual
    X_train, X_test, y_train, y_test = train_test_split(X, y["CMG_TEMT"], test_size=0.15, random_state=i)
    X_train_2, X_val, y_train_2, y_val = train_test_split(X_train, y_train, test_size=0.15, random_state=i)
        
    # Combinar conjuntos de entrenamiento y validación
    X_combined = pd.concat([X_train_2, X_val], axis=0)
    y_combined = pd.concat([y_train_2, y_val], axis=0)
    cat_model.fit(X_combined, y_combined)
        
    # Realizar predicción y calcular el MSE
    y_pred = cat_model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    mse_targets_CMG_p.append(mse)
        
    # If you want to sort the features by importance
    result = permutation_importance(cat_model, X_test, y_test, n_repeats=20)
    perm_imp = result.importances_mean
    
    sorted_indices = np.argsort(perm_imp)[::-1]
    sorted_importances = perm_imp[sorted_indices]
    sorted_features = X_train.columns[sorted_indices]

    permutation_importance_cat_CMG_p.append(perm_imp)
    permutation_importance_orden_cat_CMG_p.append(sorted_importances)


resultsagg_cat_p["CMG"] = mse_targets_CMG_p

In [54]:
resultsagg_cat_p.describe()

,CMLR,CMN,CMG
count,50.000000,50.000000,50.000000
mean,7.156769,13.461028,0.996132
std,1.021280,1.942535,0.150545
min,5.366218,9.403582,0.766755
25%,6.464744,12.105778,0.857633
50%,7.031789,13.214196,0.989700
75%,7.823694,14.294847,1.108294
max,9.451270,18.955833,1.375368


In [55]:
features_rf_CMLR_p = pd.DataFrame(data=permutation_importance_cat_CMLR_p, columns= columnas)
features_rf_CMLR_p.describe()

,Edad (años),Curso,Tipo EE,Depen. EE,MTV_WM,MTVE_ Torpo,FC_DCCS
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,0.072481,0.023413,0.027087,0.292675,0.010508,0.025473,0.083802
std,0.033770,0.014223,0.017771,0.038469,0.016732,0.017322,0.024011
min,-0.002783,-0.015574,-0.017238,0.200824,-0.050929,-0.011065,0.006259
25%,0.052205,0.014896,0.012509,0.273639,0.005683,0.012448,0.070593
50%,0.076334,0.025202,0.025814,0.293698,0.013273,0.024469,0.086645
75%,0.096451,0.032730,0.039020,0.319008,0.022248,0.035673,0.100355
max,0.143103,0.056830,0.068957,0.375004,0.039563,0.061266,0.126861


In [56]:
features_cat_CMN_p = pd.DataFrame(data=permutation_importance_cat_CMN_p, columns= columnas)
features_cat_CMN_p.describe()

,Edad (años),Curso,Tipo EE,Depen. EE,MTV_WM,MTVE_ Torpo,FC_DCCS
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,0.089237,0.019624,0.032841,0.416213,0.016298,0.010041,0.047112
std,0.021646,0.013308,0.014376,0.050221,0.013894,0.008819,0.017379
min,0.031689,-0.016216,0.004948,0.285314,-0.013650,-0.014012,0.004077
25%,0.077010,0.013125,0.024804,0.396732,0.004074,0.005463,0.036932
50%,0.090707,0.020705,0.032620,0.420537,0.019566,0.010620,0.048957
75%,0.103616,0.026523,0.040019,0.452077,0.026877,0.017000,0.057408
max,0.130945,0.051404,0.061052,0.535006,0.037713,0.022479,0.094294


In [57]:
features_cat_CMG_p = pd.DataFrame(data=permutation_importance_cat_CMG_p, columns= columnas)
features_cat_CMG_p.describe()

,Edad (años),Curso,Tipo EE,Depen. EE,MTV_WM,MTVE_ Torpo,FC_DCCS
count,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000,50.000000
mean,0.438864,0.034813,0.033895,0.345849,0.019888,0.000102,0.072439
std,0.089360,0.021298,0.020711,0.058270,0.025230,0.010193,0.030459
min,0.227218,-0.020688,-0.016911,0.234212,-0.032588,-0.023659,0.014755
25%,0.386435,0.022247,0.019405,0.303458,0.000574,-0.004745,0.053128
50%,0.455215,0.034420,0.032246,0.334111,0.017010,0.000196,0.065593
75%,0.509262,0.045252,0.049442,0.381773,0.036024,0.007710,0.098302
max,0.591517,0.088884,0.077003,0.490629,0.076351,0.018186,0.134632


In [58]:
promedio_CMLR = np.mean(permutation_importance_cat_CMLR_p, axis=0)
promedio_CMG = np.mean(permutation_importance_cat_CMG_p, axis=0)
promedio_CMN = np.mean(permutation_importance_cat_CMN_p, axis=0)

In [59]:
# Convertir cada lista a un array de NumPy. La forma resultante será (50, 12)
arr1 = np.array(permutation_importance_cat_CMLR_p)
arr2 = np.array(permutation_importance_cat_CMG_p)
arr3 = np.array(permutation_importance_cat_CMN_p)

# Sumar los arrays de forma element-wise y dividir entre 3
promedio_global = (arr1 + arr2 + arr3) / 3

promedio_final = np.mean(promedio_global, axis=0)

In [60]:
# Agregar las importancias en un ranking global: se promedian las importancias de los 3 targets para cada feature
importancia_global = (promedio_CMLR + promedio_CMG + promedio_CMN) / 3

print("Importancia promedio target CMLR:", promedio_CMLR)
print("Importancia promedio target CMG:", promedio_CMG)
print("Importancia promedio target CMN:", promedio_CMN)
print("Importancia global agregada:", importancia_global)

Importancia promedio target CMLR: [0.07248107 0.02341313 0.02708664 0.29267504 0.01050848 0.02547254
 0.08380181]
Importancia promedio target CMG: [4.38864154e-01 3.48131831e-02 3.38953819e-02 3.45849328e-01
 1.98881308e-02 1.01936003e-04 7.24387213e-02]
Importancia promedio target CMN: [0.08923675 0.01962388 0.03284084 0.4162134  0.0162981  0.01004062
 0.04711195]
Importancia global agregada: [0.20019399 0.02595006 0.03127429 0.35157926 0.0155649  0.0118717
 0.06778416]


In [61]:
# Convertir las importancias a rankings
# Se utiliza rank con ascending=False, pues se asume que un valor mayor de importancia implica mejor posición.
ranking_CMLR = pd.Series(promedio_CMLR).rank(ascending=False).values
ranking_CMN = pd.Series(promedio_CMN).rank(ascending=False).values
ranking_CMG = pd.Series(promedio_CMG).rank(ascending=False).values

ranking_global = pd.Series(importancia_global).rank(ascending=False).values

print("Edad (años), Tipo EE, Depen. EE, MTV_WM, MTVE_Torpo,FC_DCCS")
print("\nRanking target CMLR:", ranking_CMLR)
print("Ranking target CMN:", ranking_CMN)
print("Ranking target CMG:", ranking_CMG)

print("Ranking global:", ranking_global)

#"Edad (años)", "Tipo EE", "Depen. EE", "MTV_WM", 'MTVE_ Torpo',"FC_DCCS"
# Paso 3: Validación estadística con el Test de Friedman
# El test evalúa si existen diferencias significativas entre los rankings obtenidos de cada target.
# Aquí, cada array (ranking_A, ranking_B, ranking_C) representa las posiciones asignadas a cada feature.
stat, p_value = friedmanchisquare(ranking_CMLR, ranking_CMG, ranking_CMN)
print("\nTest de Friedman:")
print("Estadístico =", stat)
print("p-valor =", p_value)

Edad (años), Tipo EE, Depen. EE, MTV_WM, MTVE_Torpo,FC_DCCS

Ranking target CMLR: [3. 6. 4. 1. 7. 5. 2.]
Ranking target CMN: [2. 5. 4. 1. 6. 7. 3.]
Ranking target CMG: [1. 4. 5. 2. 6. 7. 3.]
Ranking global: [2. 5. 4. 1. 6. 7. 3.]

Test de Friedman:
Estadístico = 0.08695652173912796
p-valor = 0.9574533680683821


<p style="font-size:20px; font-weight: bold;">Wilconxon CatBoost CMLR.</p>

In [63]:
stat, p_valor = wilcoxon(features_rf_CMLR_p["MTV_WM"], features_rf_CMLR_p["FC_DCCS"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [64]:
stat, p_valor = wilcoxon(features_rf_CMLR_p["FC_DCCS"], features_rf_CMLR_p["Edad (años)"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 461.0
p-valor: 0.0894381024777875


In [65]:
stat, p_valor = wilcoxon(features_rf_CMLR_p["Edad (años)"], features_rf_CMLR_p["Depen. EE"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [66]:
stat, p_valor = wilcoxon(features_rf_CMLR_p["Depen. EE"], features_rf_CMLR_p["Tipo EE"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [67]:
stat, p_valor = wilcoxon(features_rf_CMLR_p["Tipo EE"], features_rf_CMLR_p["MTVE_ Torpo"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 599.0
p-valor: 0.7160933233749613


<p style="font-size:20px; font-weight: bold;">Wilconxon CatBoost CMG.</p>

In [69]:
stat, p_valor = wilcoxon(features_cat_CMG_p["MTV_WM"], features_cat_CMG_p["Edad (años)"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [70]:
stat, p_valor = wilcoxon(features_cat_CMG_p["MTV_WM"], features_cat_CMG_p["FC_DCCS"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [71]:
stat, p_valor = wilcoxon(features_cat_CMG_p["Depen. EE"], features_cat_CMG_p["FC_DCCS"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [72]:
stat, p_valor = wilcoxon(features_cat_CMG_p["Depen. EE"], features_cat_CMG_p["Tipo EE"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [73]:
stat, p_valor = wilcoxon(features_cat_CMG_p["Tipo EE"], features_cat_CMG_p["MTVE_ Torpo"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 17.0
p-valor: 3.6770586575585185e-13


<p style="font-size:20px; font-weight: bold;">Wilconxon CatBoost CMN.</p>

In [75]:
stat, p_valor = wilcoxon(features_cat_CMN_p["MTV_WM"], features_cat_CMN_p["Edad (años)"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 1.0
p-valor: 3.552713678800501e-15


In [76]:
stat, p_valor = wilcoxon(features_cat_CMN_p["Edad (años)"], features_cat_CMN_p["FC_DCCS"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 35.0
p-valor: 7.65609797781508e-12


In [77]:
stat, p_valor = wilcoxon(features_cat_CMN_p["Depen. EE"], features_cat_CMN_p["FC_DCCS"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [78]:
stat, p_valor = wilcoxon(features_cat_CMN_p["Depen. EE"], features_cat_CMN_p["Tipo EE"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 0.0
p-valor: 1.7763568394002505e-15


In [79]:
stat, p_valor = wilcoxon(features_cat_CMN_p["Tipo EE"], features_cat_CMN_p["MTVE_ Torpo"])

print("Estadístico de Wilcoxon:", stat)
print("p-valor:", p_valor)

Estadístico de Wilcoxon: 11.0
p-valor: 9.769962616701378e-14


In [80]:
# Aplicamos la corrección de Bonferroni.
# multipletests multiplica cada p-valor por el número de comparaciones o ajusta el nivel alpha.
p_values = [1.7763568394002505e-15, 0.1781010477908609, 1.2871659293978155e-10, 0.22231487615329293, 0.0036632546526647047, 2.305757291765076e-07,
           1.7763568394002505e-15, 1.370373858833318e-08, 0.7813934753886187, 0.002996250182501825, 1.7763568394002505e-15, 6.535714192068554e-10,
            9.898605478397826e-06, 1.1645574410579229e-05, 0.4723763440876976]

rej, pvals_corr, alphacSidak, alphacBonf = multipletests(p_values, alpha=0.05, method='bonferroni')

print("\nP-valores ajustados con Bonferroni:")
print(pvals_corr)

print("\nHipótesis nula rechazada (True significa que se rechaza):")
print(rej)


P-valores ajustados con Bonferroni:
[2.66453526e-14 1.00000000e+00 1.93074889e-09 1.00000000e+00
 5.49488198e-02 3.45863594e-06 2.66453526e-14 2.05556079e-07
 1.00000000e+00 4.49437527e-02 2.66453526e-14 9.80357129e-09
 1.48479082e-04 1.74683616e-04 1.00000000e+00]

Hipótesis nula rechazada (True significa que se rechaza):
[ True False  True False False  True  True  True False  True  True  True
  True  True False]
